In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

# vector DB : Chroma / Pinecone
- Chroma : 인메모리 vector DB,local vector DB, collection_name
- [Pinecone](http://www.pinecone.io) : cloud vector DB, index_name
    - API KEY의 .env 설정 필요
    
# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


# 1. knowledge base 구성을 위한 데이터 생성

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n\n\n", "\n\n", "\n", " "] # 분할 기준 우선순위
)

documents = loader.load_and_split(text_splitter=text_splitter)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19708\1660385314.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [2]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

import os
load_dotenv()

embedding = UpstageEmbeddings(
    model='solar-embedding-1-large-query'
)

In [3]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
pc = Pinecone()

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# %%time
# # 데이터 첫 저장 시
# index_name = 'tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )

CPU times: total: 15 s
Wall time: 27.5 s


In [4]:
%%time
# vectorDB 로드 시
index_name = 'tax-index'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 156 ms
Wall time: 1.04 s


# 2. 답변 생성을 위한 Retrieval

In [6]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(
    query=query, 
    k=2 # 기본값 : 4
)

In [7]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [8]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model='solar-pro4')

In [9]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가 입니다
- 다음의 [context]를 참조하여 사용자의 질문에 답변하세요
[context]
{retrieved_doc}
질문 : {query}"""

ai_msg = llm.invoke(prompt)